<div class="alert alert-warning">

# PS 12 — Associative Memory and the Hopfield Network

How does the brain retrieve a memory from a partial or noisy cue? In this problem set you will implement and explore the **Hopfield network** — a recurrent neural network that stores memories as stable patterns of activity and retrieves them by a simple local update rule.

## The model

An $N$-neuron network stores $P$ binary patterns $\xi^\mu \in \{-1,+1\}^N$. Weights are set by the **Hebbian rule**:

$$W_{ij} = \frac{1}{N}\sum_{\mu=1}^{P} \xi^\mu_i\,\xi^\mu_j \qquad (W_{ii} = 0)$$

During retrieval, the network starts from a cue and updates synchronously until convergence:

$$s_i(t+1) = \operatorname{sign}\!\left(\sum_j W_{ij}\,s_j(t)\right)$$

Each update is guaranteed not to increase the **energy**:

$$E(s) = -\frac{1}{2}\,s^\top W s$$

so the network always converges to a local energy minimum — ideally a stored pattern.

We will explore:
1. How the network stores and retrieves memories from noisy cues
2. What happens as the number of stored patterns grows (capacity and false memories)
3. Why similar patterns interfere, and what this implies for memory in the brain

</div>

In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# ── PROVIDED ────────────────────────────────────────────────────────────────
N = 100   # 100 neurons arranged as a 10x10 grid


def random_patterns(P, a=0.5, seed=None):
    """Generate P random patterns of length N with fraction a of +1 entries."""
    rng = np.random.default_rng(seed)
    return rng.choice([-1, 1], size=(P, N), p=[1 - a, a])


def add_noise(pattern, noise, seed=None):
    """Flip each bit independently with probability `noise`."""
    rng = np.random.default_rng(seed)
    s   = pattern.copy().astype(float)
    s[rng.random(N) < noise] *= -1
    return s


def overlap(s, pattern):
    """Normalised overlap in [-1, 1]. Returns 1 if s == pattern."""
    return float(np.dot(s, pattern)) / N


def show_patterns(pats, titles, suptitle=''):
    """Display binary patterns side by side as 10x10 images."""
    fig, axes = plt.subplots(1, len(pats), figsize=(3 * len(pats), 3))
    if len(pats) == 1:
        axes = [axes]
    for ax, p, t in zip(axes, pats, titles):
        ax.imshow(p.reshape(10, 10), cmap='gray_r', vmin=-1, vmax=1)
        ax.set_title(t, fontsize=9)
        ax.axis('off')
    if suptitle:
        plt.suptitle(suptitle, y=1.02)
    plt.tight_layout()
    plt.show()


print(f"Helpers loaded. N = {N}")

Helpers loaded. N = 100


---

# 1. Storing and Retrieving Memories

<div class="alert alert-success">

## Exercise 1A

Implement `hebbian_weights(patterns)` — compute the $N \times N$ weight matrix from the formula above (set the diagonal to zero).

    
$$W_{ij} = \frac{1}{N}\sum_{\mu=1}^{P} \xi^\mu_i\,\xi^\mu_j \qquad (W_{ii} = 0)$$
</div>

In [1]:
def hebbian_weights(patterns):
    """
    patterns         - (P, N) binary patterns
    
    Hebbian weight matrix: W  (N, N), created by multipyling two matrices using @
    patterns.T (N, P) and patterns (P, N), divided by N
    
    Then set diagnoal = 0 using np.fill_diagonal()

    Returns: W
    """
    #(TODO: replace pass with your implementation)
    pass

<div class="alert alert-success">

## Exercise 1B
    
Implement `update(W, state, n_steps=20)` — run synchronous updates (formula below) until convergence or `n_steps` iterations. Return the final state and a list of all intermediate states (for plotting the energy trajectory).

$$s_i(t+1) = \operatorname{sign}\!\left(\sum_j W_{ij}\,s_j(t)\right)$$

</div>

In [ ]:
def update(W, state, n_steps=20):
    """
    Synchronous Hopfield update until convergence.
    Returns (final_state, history) where history is a list of states.
    """
    
    s = state.copy().astype(float) # make a float copy of the initial state
    history = [s.copy()] # store the initial state
    for _ in range(n_steps):
        
        s_new = #(TODO: implementation of synchronous updates)

        s_new[s_new == 0] = 1.0   # break sign(0) ties, replace 0s with +1
        history.append(s_new.copy()) # save the new state
        if np.all(s_new == s): # stop if state no longer changes
            break
        s = s_new # otherwise continue from new state
    
    return s, history # return final state and full history

<div class="alert alert-success">

Then run a sanity check: store 3 random patterns and verify that each stored pattern is a fixed point of the network (overlap with itself after update $\approx 1.0$).

</div>

In [ ]:

# Sanity check: stored patterns should be fixed points
P = 3
patterns = random_patterns(P, seed=0)
W = hebbian_weights(patterns)
for mu in range(P):
    final, _ = update(W, patterns[mu])
    print(f"Pattern {mu}: overlap after update = {overlap(final, patterns[mu]):.3f}")

<div class="alert alert-success">

## Exercise 1C
    
Implement `energy(W, state)` — return $E(s) = -\frac{1}{2} s^\top W s$.

</div>

In [ ]:
def energy(W, state):
    """
    Returns: Hopfield energy E(s) = -0.5 * s^T W s.
    """
    pass

<div class="alert alert-success">

## Exercise 1D

Use the patterns you just created. Create a noisy cue by flipping 30% of the bits of the first pattern. Run the network from the cue and use `show_patterns` to display: (i) the original pattern, (ii) the noisy cue, and (iii) the retrieved state.

</div>

In [ ]:
# Exercise 1D: retrieve from a 30% noisy cue and show original / cue / retrieved
cue = add_noise(patterns[0], noise=0.30, seed=1)
#(TODO: Run the network from the cue; store the retrieved state and history)

show_patterns(
    [patterns[0], cue, retrieved],
    ['Original',
     'Cue (30% noise)',
     f'Retrieved (overlap={overlap(retrieved, patterns[0]):.2f})'])

<div class="alert alert-success">

## Exercise 1E    
    
Plot the energy $E(s(t))$ over iterations during the retrieval in **Exercise 1D**. What do you observe?

</div>

In [ ]:
# Exercise 1E: energy trajectory during retrieval
energies = [energy(W, s) for s in history]

fig, ax = plt.subplots(figsize=(5, 3))
#(TODO: use ax.plot to plot energy trajectories)
#(TODO: set proper x and y labels)
#(TODO: set the title, describe what you observed)
plt.tight_layout()
plt.show()

---

# 2. Memory Capacity and False Memories

The Hopfield network cannot store arbitrarily many patterns. Beyond a critical load ratio $\alpha = P/N \approx 0.138$, retrieval fails catastrophically. When the network is overloaded it can also get stuck in **spurious attractors** — stable states that do not correspond to any stored pattern, analogous to false or distorted memories.


<div class="alert alert-success">

## Exercise 2A  

For $P \in \{2, 4, 6, \ldots, 26\}$ patterns ($N = 100$): store $P$ random patterns, present the first pattern with 10% noise, and run the network. Average retrieval accuracy (overlap $> 0.9$ counts as correct) over 30 independent repetitions. Plot accuracy vs. $\alpha = P/N$ and add a vertical line at the theoretical limit $\alpha \approx 0.138$.
    
</div>

In [ ]:
# Exercise 2A: retrieval accuracy vs. load ratio
P_values  = np.arange(2, 28, 2)
n_trials  = 30
accuracy  = []
np.random.seed(0)

for P in P_values:
    hits = 0
    for t in range(n_trials):
        pats  = random_patterns(P, seed=t * 1000)
        W_t   = #(TODO: store patterns into hebbian weights)
        cue   = #(TODO: add 10% noise to the first pattern, set seed number as t)
        final, _ = update(W_t, cue)
        if overlap(final, pats[0]) > 0.9:
            hits += 1
    accuracy.append(hits / n_trials)

fig, ax = plt.subplots(figsize=(6, 4))
#(TODO: plot accuracy vs. alpha = P/N)
ax.axvline(0.138, color='tomato', linestyle='--', label=r'Theoretical limit ($\alpha\approx 0.138$)')
#(TODO: set proper x and y labels)
#(TODO: set the title, describe what you observed)
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 2B

In 1–2 sentences, describe the shape of the curve. Is the breakdown gradual or sudden?
    
</div>

**2B. Answer:** *(Write your answer here.)*

<div class="alert alert-success">

## Exercise 2C

Use an overloaded network ($P = 22$, $\alpha \approx 0.22$). Start from 200 random initial states and run to convergence. For each final state, compute the maximum overlap with any stored pattern. Plot a histogram of these maximum overlaps. What fraction of runs end in a spurious attractor (max overlap $< 0.9$)?
    
</div>

In [ ]:
# Exercise 2C: spurious attractors in an overloaded network (P=22)
P_over   = 22          # alpha = 0.22
patterns_over = random_patterns(P_over, seed=42)
W_over   = #(TODO: store patterns_over into hebbian weights)

n_tests      = 200
max_overlaps = []
np.random.seed(1)

for t in range(n_tests):
    
    s0 = #(TODO: create a random inital state (N,))
    final, _ = #(TODO: use W_over to update s0)
    
    max_ol = max(abs(overlap(final, patterns_over[mu])) for mu in range(P_over))
    max_overlaps.append(max_ol)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(max_overlaps, bins=20, color='steelblue', edgecolor='k')
ax.axvline(0.9, color='tomato', linestyle='--', label='Stored-pattern threshold (0.9)')
ax.set_xlabel('Max overlap with any stored pattern')
ax.set_ylabel('Count (200 random starts)')
ax.set_title(f'Spurious attractors (P={P_over}, N={N}, ' + r'$\alpha$' + f'={P_over/N:.2f})')
ax.legend()
plt.tight_layout()
plt.show()

n_spurious = sum(o < 0.9 for o in max_overlaps)
print(f'{n_spurious}/{n_tests} trials settled in a spurious attractor')

<div class="alert alert-success">

## Exercise 2D

In 1–2 sentences, what do spurious attractors represent as a model of human memory?
    
</div>

**2D. Answer:** *(Write your answer here.)*

---

# 3. Pattern Completion vs. Pattern Separation

The Hopfield network is naturally a **pattern completion** device: it can recover a full memory from a degraded cue. But this ability comes at a cost — similar memories interfere with each other, reducing capacity and causing confusions.

The mammalian hippocampus is thought to balance these demands across two subregions:

- **CA3** (dense recurrent connections) — performs pattern completion, like a Hopfield network.
- **Dentate gyrus (DG)** (sparse, orthogonalised representations) — performs **pattern separation**: transforming similar inputs into more distinct codes before they are stored in CA3.

In this section you will (A) characterise the network's noise tolerance, then (B) show that storing similar patterns dramatically reduces capacity, directly motivating the need for a separation mechanism.


<div class="alert alert-success">

## Exercise 3A

With $P = 3$ random patterns, measure retrieval accuracy as a function of cue noise level (0% – 50% bits flipped, 15 levels). Average over 50 independent pattern sets. Plot the completion curve.

</div>

In [ ]:
# Exercise 3A: noise tolerance curve (P=3, 15 noise levels from 0 to 0.5)
noise_levels    = #(TODO: 0%-15%, 15 levels)
P_fixed         = 3
n_trials_3a     = 50
accuracy_noise  = []

np.random.seed(2)
for #(TODO: varying across 15 noise levels)
    hits = 0
    for #(TODO: varying across 50 independent pattern sets)
        pats  = random_patterns(P_fixed, seed=t * 100)
        W_t   = hebbian_weights(pats)
        cue   = add_noise(pats[0], noise=noise, seed=t)
        final, _ = update(W_t, cue)
        if overlap(final, pats[0]) > 0.9:
            hits += 1
    accuracy_noise.append(hits / n_trials_3a)

fig, ax = plt.subplots(figsize=(6, 4))
#(TODO: plot accuracy vs. noise levels)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Chance')
#(TODO: set proper x and y labels)
ax.set_title(f'Pattern completion: noise tolerance (P={P_fixed})')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 3B

Compare capacity for two types of patterns:
- **Random** (pairwise overlap $\approx 0$): generated as before.
- **Similar** (pairwise overlap $\approx 0.13$): each pattern is produced by flipping 32% of bits from a shared base.

For each type, plot retrieval accuracy vs. $\alpha = P/N$ (use $P \in \{2, 4, \ldots, 26\}$, 20 repetitions, 10% noise cue). Overlay both curves.

</div>

In [ ]:
# Provided: helper to generate similar patterns
def similar_patterns(P, flip=0.32, seed=0):
    """
    Generate P patterns derived from a shared base by flipping `flip` fraction
    of bits. Pairwise overlap ~ (1 - 2*flip)^2 ~ 0.13 for flip=0.32.
    """
    rng  = np.random.default_rng(seed)
    base = rng.choice([-1, 1], N)
    pats = []
    for _ in range(P):
        p = base.copy()
        p[rng.random(N) < flip] *= -1
        pats.append(p)
    return np.array(pats)


# Exercise 3B: capacity curve for random vs. similar patterns
P_values   = np.arange(2, 28, 2)
n_trials   = 20
noise_test = 0.10

acc_random  = []
acc_similar = []
np.random.seed(3)

for P in P_values:
    hits_r = hits_s = 0
    for t in range(n_trials):
        # Random patterns
        pats_r  = random_patterns(P, seed=t * 1000)
        W_r     = hebbian_weights(pats_r)
        cue_r   = add_noise(pats_r[0], noise=noise_test, seed=t)
        if overlap(update(W_r, cue_r)[0], pats_r[0]) > 0.9:
            hits_r += 1
        # Similar patterns
        pats_s  = #(TODO: generate patterns)
        W_s     = #(TODO: store patterns into hebbian weights)
        cue_s   = #(TODO: add noise to the first pattern, set seed number as t)
        if overlap(update(W_s, cue_s)[0], pats_s[0]) > 0.9:
            hits_s += 1
    acc_random.append(#(TODO: calculate accuracy))
    acc_similar.append(#(TODO: calculate accuracy)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(P_values / N, acc_random,  'o-', color='steelblue', label='Random patterns')
ax.plot(P_values / N, acc_similar, 's--', color='tomato',   label='Similar patterns')
ax.axvline(0.138, color='gray', linestyle=':', label=r'Theoretical limit ($\alpha\approx0.138$)')
ax.set_xlabel(r'Load ratio $\alpha = P/N$')
ax.set_ylabel('Retrieval accuracy')
ax.set_title('Capacity: random vs. similar patterns')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 3C

In 2–3 sentences, explain what you observe in **Exercise B** and connect it to the CA3/DG distinction.

</div>

**3C. Answer:** *(Write your answer here.)*

---

# 4. Implementing Pattern Separation

Section 3 showed that *similar* patterns dramatically reduce Hopfield capacity. Here we implement a minimal model of the **dentate gyrus (DG)** that restores it.

## The DG model

The DG receives input from the entorhinal cortex (EC) and projects to CA3. We model it as a two-step transform applied to each pattern:

1. **Random expansion**: project the $N$-bit EC pattern to $N_\text{DG} \gg N$ DG neurons via a fixed random matrix.
2. **Competitive inhibition**: subtract the per-neuron mean activation (shared across all stored patterns) before taking the sign. This removes the contribution of the common *base* pattern shared by similar inputs, leaving only each pattern's unique bits to drive distinct DG codes.

$$\xi^\mu_\text{DG} = \text{sign}\!\bigl(W_\text{proj}\,\xi^\mu - \bar{a}\bigr), \qquad \bar{a} = \frac{1}{P}\sum_\mu W_\text{proj}\,\xi^\mu$$

The bias $\bar{a}$ must be saved during storage and reused when encoding retrieval cues.


In [6]:
# ── PROVIDED for Section 4 ─────────────────────────────────────────────────
N_DG  = 300   # DG expansion (3× larger than CA3)

np.random.seed(99)
W_proj = np.random.randn(N_DG, N) / np.sqrt(N)   # fixed random projection


def dg_encode(patterns):
    """
    Encode a batch of P patterns through the dentate-gyrus model.

    Steps:
      1. Random projection to N_DG dimensions.
      2. Subtract the per-neuron mean (removes the shared base-pattern
         component via competitive inhibition).
      3. Take sign to produce ±1 codes.

    Returns
    -------
    codes : (P, N_DG) array of ±1
    bias  : (N_DG,) per-neuron mean — save this and pass to dg_encode_cue
            so that retrieval cues are encoded in the same way.
    """
    act  = patterns @ W_proj.T          # (P, N_DG)
    bias = act.mean(axis=0)             # shared component across stored patterns
    codes = np.sign(act - bias)
    codes[codes == 0] = 1.0
    return codes, bias


def dg_encode_cue(pattern, bias):
    """Encode a single retrieval cue using the bias saved during storage."""
    act  = pattern @ W_proj.T
    code = np.sign(act - bias)
    code[code == 0] = 1.0
    return code


print(f"DG projection ready: N={N} → N_DG={N_DG}")

DG projection ready: N=100 → N_DG=300


<div class="alert alert-success">

## Exercise 4A

Generate $P = 20$ highly similar patterns (`flip=0.10`, pairwise overlap $\approx 0.64$). Encode them through the DG. Compute all pairwise overlaps before and after, and display the two distributions as side-by-side histograms on a shared x-axis.

</div>

In [ ]:
# Exercise 4A: pairwise overlap before and after DG separation
from itertools import combinations

P_demo = 20
pats_hs = similar_patterns(P_demo, flip=0.10, seed=42)
codes_dg, bias_demo = dg_encode(pats_hs)

pairs = list(combinations(range(P_demo), 2))
ov_before = [float(pats_hs[i] @ pats_hs[j]) / N     for i, j in pairs]
ov_after  = #(TODO: calculate pairwise overlaps for DG codes)

print(f"Mean pairwise overlap  BEFORE: {np.mean(ov_before):.3f}")
print(f"Mean pairwise overlap  AFTER:  {np.mean(ov_after):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
bins = np.linspace(-0.5, 1.0, 31)

axes[0].hist(ov_before, bins=bins, color='tomato',    edgecolor='k', alpha=0.8)
axes[0].axvline(0, color='k', linestyle='--', linewidth=1)
axes[0].set_xlabel('Pairwise overlap')
axes[0].set_ylabel('Number of pairs')
axes[0].set_title('EC patterns (before separation)')

axes[1].hist(ov_after, bins=bins, color='steelblue', edgecolor='k', alpha=0.8)
axes[1].axvline(0, color='k', linestyle='--', linewidth=1)
axes[1].set_xlabel('Pairwise overlap')
axes[1].set_title('DG codes (after separation)')

plt.suptitle(f'Pairwise overlaps: P={P_demo} similar patterns (flip=0.10)', y=1.02)
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 4B

Extend the capacity comparison from Exercise 3B with a third curve: similar patterns (`flip=0.10`) after DG separation, stored in a $N_\text{DG}$-neuron Hopfield network. Use the same protocol (20 repetitions, 10% noise cue, $P \in \{2, 4, \ldots, 26\}$). Plot all three curves.

</div>

In [ ]:
# Exercise 4B: capacity comparison with DG separation
P_values   = np.arange(2, 28, 2)
n_trials   = 20
noise_test = 0.10

acc_rand4, acc_sim4, acc_dg4 = [], [], []
np.random.seed(7)

for P in P_values:
    hr = hs = hd = 0
    for t in range(n_trials):
        # Random patterns
        pr = random_patterns(P, seed=t * 1000)
        Wr = hebbian_weights(pr)
        if overlap(update(Wr, add_noise(pr[0], noise_test, seed=t))[0], pr[0]) > 0.9:
            hr += 1

        # Highly similar patterns (flip=0.10)
        ps = similar_patterns(P, flip=0.10, seed=t * 1000)
        Ws = hebbian_weights(ps)
        if overlap(update(Ws, add_noise(ps[0], noise_test, seed=t))[0], ps[0]) > 0.9:
            hs += 1

        # DG-separated similar patterns
        cd, bi = #TODO(Encode similar patterns into DG codes)
        Wd     = cd.T @ cd / N_DG
        np.fill_diagonal(Wd, 0)
        cue_d  = dg_encode_cue(add_noise(ps[0], noise_test, seed=t), bi)
        final_d, _ = update(Wd, cue_d)
        if float(final_d @ cd[0]) / N_DG > 0.9:
            hd += 1

    acc_rand4.append(hr / n_trials)
    #TODO(Calculate and append acc_sim4)
    #TODO(Calculate and append acc_dg4)

fig, ax = plt.subplots(figsize=(7, 4))
alphas = P_values / N
ax.plot(alphas, acc_rand4, 'o-',  color='steelblue', label='Random patterns')
ax.plot(alphas, acc_sim4,  's--', color='tomato',    label='Similar patterns (no DG)')
ax.plot(alphas, acc_dg4,   '^-',  color='seagreen',  label='Similar + DG separation')
ax.axvline(0.138, color='gray', linestyle=':', label=r'Theoretical limit ($\alpha\approx0.138$)')
ax.set_xlabel(r'Load ratio $\alpha = P/N$')
ax.set_ylabel('Retrieval accuracy')
ax.set_title('Capacity: DG separation restores performance for similar patterns')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 4C

In 2–3 sentences: what do these results tell you about the division of labour between the dentate gyrus and CA3?

</div>

**4C. Answer:** *(Write your answer here.)*

---

<div class="alert alert-warning">


When you are done, **restart the kernel and re-run the whole notebook** (`Kernel -> Restart & Run All`) to make sure everything runs without errors.

</div>